In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import optuna

In [2]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

In [3]:
def model_tuning(X_train, X_test, y_train, y_test, trial, model_class, seed=0):
    # Define the hyperparameters
    if model_class == LinearRegression:
        param = {
        }
    elif model_class == KNeighborsRegressor:
        param = {
        'n_neighbors': trial.suggest_int('n_neighbors', 1, 10),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'algorithm': trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute']),
        'leaf_size': trial.suggest_int('leaf_size', 20, 50),
        'p': trial.suggest_int('p', 1, 2),
    }
    elif model_class == DecisionTreeRegressor:
        param = {
        'random_state': seed,
        'max_depth': trial.suggest_int('max_depth', 1, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 5),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 2, 100),
        'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 1.0),
    }
    elif model_class == RandomForestRegressor:
        param = {
        'random_state': seed,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 5),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    }
    elif model_class == XGBRegressor:
        param = {
        'random_state': seed,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 5),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        }
    elif model_class == AdaBoostRegressor:
        param = {
        'random_state': seed,
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 1),
        'loss': trial.suggest_categorical('loss', ['linear', 'square', 'exponential']),
    }
    elif model_class == GradientBoostingRegressor:
        param = {
        'random_state': seed,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 5),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    }

    # Create the model
    models = [model_class(**param)]
    model = models[0]
    model.fit(X_train, y_train)
    
    # Generate forecasts
    y_pred = model.predict(X_test)
    # Calculate the mean absolute error
    mape = mean_absolute_percentage_error(y_test, y_pred)

    return mape

In [4]:
data = pd.read_csv('data/01_Raw/Wine_Quality_Data.csv')
data['color'] = data['color'].map({'white': 0, 'red': 1})
data.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,color
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,1
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1


In [5]:
X_train, X_test, y_train, y_test = train_test_split(data.drop('residual_sugar', axis=1), data['residual_sugar'], test_size=0.3, random_state=42)

In [6]:
model_classes = [LinearRegression, 
                 KNeighborsRegressor, 
                 DecisionTreeRegressor, 
                 RandomForestRegressor, 
                 XGBRegressor, 
                 AdaBoostRegressor, 
                 GradientBoostingRegressor]
studies = {}
models = {}
for model_class in model_classes:
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: model_tuning(X_train, X_test, y_train, y_test, trial, model_class, seed=0), n_trials=4)
    studies[model_class.__name__] = study
    models[model_class.__name__] = [model_class(**study.best_params)]

[I 2024-03-19 17:13:58,601] A new study created in memory with name: no-name-ebb7b47e-33f3-4c74-ad27-662202b358ee
[I 2024-03-19 17:13:58,601] Trial 0 finished with value: 0.40199617118055253 and parameters: {}. Best is trial 0 with value: 0.40199617118055253.
[I 2024-03-19 17:13:58,612] Trial 1 finished with value: 0.40199617118055253 and parameters: {}. Best is trial 0 with value: 0.40199617118055253.
[I 2024-03-19 17:13:58,616] Trial 2 finished with value: 0.40199617118055253 and parameters: {}. Best is trial 0 with value: 0.40199617118055253.
[I 2024-03-19 17:13:58,616] Trial 3 finished with value: 0.40199617118055253 and parameters: {}. Best is trial 0 with value: 0.40199617118055253.
[I 2024-03-19 17:13:58,616] A new study created in memory with name: no-name-12c06c6c-a33e-43ff-97c4-1263027667a2


[I 2024-03-19 17:13:58,795] Trial 0 finished with value: 0.7005006558554452 and parameters: {'n_neighbors': 6, 'weights': 'distance', 'algorithm': 'brute', 'leaf_size': 24, 'p': 1}. Best is trial 0 with value: 0.7005006558554452.
[I 2024-03-19 17:13:58,821] Trial 1 finished with value: 0.7004885021048617 and parameters: {'n_neighbors': 6, 'weights': 'distance', 'algorithm': 'auto', 'leaf_size': 40, 'p': 1}. Best is trial 1 with value: 0.7004885021048617.
[I 2024-03-19 17:13:58,834] Trial 2 finished with value: 0.743231942345268 and parameters: {'n_neighbors': 2, 'weights': 'distance', 'algorithm': 'kd_tree', 'leaf_size': 47, 'p': 2}. Best is trial 1 with value: 0.7004885021048617.
[I 2024-03-19 17:13:58,853] Trial 3 finished with value: 0.7193551841553709 and parameters: {'n_neighbors': 1, 'weights': 'distance', 'algorithm': 'brute', 'leaf_size': 33, 'p': 1}. Best is trial 1 with value: 0.7004885021048617.
[I 2024-03-19 17:13:58,853] A new study created in memory with name: no-name-6df

In [7]:
# Best models parameters
models

{'LinearRegression': [LinearRegression()],
 'KNeighborsRegressor': [KNeighborsRegressor(leaf_size=40, n_neighbors=6, p=1, weights='distance')],
 'DecisionTreeRegressor': [DecisionTreeRegressor(max_depth=9, max_leaf_nodes=72,
                        min_impurity_decrease=0.4899462446310521,
                        min_samples_leaf=4, min_samples_split=3)],
 'RandomForestRegressor': [RandomForestRegressor(max_depth=16, max_features='log2', min_samples_leaf=2,
                        min_samples_split=4, n_estimators=352)],
 'XGBRegressor': [XGBRegressor(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               gamma=None, grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=None, max_bin=None,
               max_cat_thresh

In [8]:
result = y_test.to_frame().reset_index(drop=True)
for model_name, model in models.items():
    model = models[model_name][0]
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    result = pd.concat([result, pd.DataFrame(y_pred, columns=[model_name])],axis=1)

In [9]:
result 

,residual_sugar,LinearRegression,KNeighborsRegressor,DecisionTreeRegressor,RandomForestRegressor,XGBRegressor,AdaBoostRegressor,GradientBoostingRegressor
0,12.80,9.972294,2.686579,6.725912,6.448237,13.213928,7.180126,8.132263
1,2.20,0.890713,8.688296,2.274427,2.904277,2.104761,4.086494,2.469204
2,7.40,6.160562,7.400000,2.918730,6.881887,7.400670,4.794026,7.461019
3,11.20,10.292324,5.528573,6.725912,8.193137,8.321177,7.473725,8.459620
4,13.90,13.423957,13.496520,14.426769,14.713574,13.510039,14.448211,14.346460
...,...,...,...,...,...,...,...,...
1945,9.65,6.407420,8.633262,2.918730,5.252209,7.965338,4.620973,6.798636
1946,1.30,2.016811,2.262995,6.725912,2.412042,2.774049,5.356698,2.079559
1947,2.50,1.864042,5.852974,2.918730,3.023748,2.668916,4.092768,2.815994
1948,2.00,0.694498,6.108141,2.918730,1.889172,1.555169,3.493254,1.233838


In [10]:
def calculate_metrics(actual, prediction):
    mape = mean_absolute_percentage_error(actual, prediction) * 100
    mae = mean_absolute_error(actual, prediction)
    mse = mean_squared_error(actual, prediction)

    metrics_df = pd.DataFrame({
        'MAPE': [mape],
        'MAE': [mae],
        'MSE': [mse]
    })

    return metrics_df

In [11]:
model = ['LinearRegression', 
         'KNeighborsRegressor', 
         'DecisionTreeRegressor', 
         'RandomForestRegressor', 
         'XGBRegressor', 
         'AdaBoostRegressor', 
         'GradientBoostingRegressor']
metrics_df = pd.DataFrame()
actual = result['residual_sugar']
for model_name in model:
    prediction = result[model_name]
    model_metrics = calculate_metrics(actual, prediction)
    model_metrics.index = [model_name]
    metrics_df = pd.concat([metrics_df, model_metrics])

metrics_df.round(2)

,MAPE,MAE,MSE
LinearRegression,40.20,1.13,2.25
KNeighborsRegressor,70.05,1.96,10.28
DecisionTreeRegressor,56.24,1.79,6.22
RandomForestRegressor,35.09,1.01,2.20
XGBRegressor,21.57,0.70,2.23
AdaBoostRegressor,82.85,2.06,5.89
GradientBoostingRegressor,22.82,0.69,1.18
